# E-Commerce Analytics & Intelligence Platform

**Scope:** advanced EDA, RFM customer segmentation, monthly sales forecasting, and Power BI exports.

**Google Colab:** open this notebook and run cells from top to bottom. The setup cell installs missing libraries and prompts you to upload `ecommerce_tableau_final.csv` if it is not already available.

All generated datasets and model files are saved in the `outputs/` folder. Use the Colab Files panel to download them.

In [ ]:
# Google Colab setup
# If the CSV is not already in the working directory, this cell will ask you to upload it.
import sys, subprocess, importlib.util
required = {"pandas":"pandas", "numpy":"numpy", "matplotlib":"matplotlib", "seaborn":"seaborn", "sklearn":"scikit-learn", "joblib":"joblib"}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor
import joblib, json
from pathlib import Path

# Create output folders
OUTPUT = Path("outputs")
for folder in ["data", "models", "reports"]:
    (OUTPUT / folder).mkdir(parents=True, exist_ok=True)

# Locate the dataset or upload it from your computer
candidates = [Path("ecommerce_tableau_final.csv"), Path("data/ecommerce_tableau_final.csv"), Path("/content/ecommerce_tableau_final.csv")]
DATA_PATH = next((p for p in candidates if p.exists()), None)
if DATA_PATH is None:
    try:
        from google.colab import files
        print("Upload ecommerce_tableau_final.csv")
        uploaded = files.upload()
        csv_files = [name for name in uploaded if name.lower().endswith(".csv")]
        if not csv_files:
            raise FileNotFoundError("No CSV file was uploaded. Please upload ecommerce_tableau_final.csv.")
        DATA_PATH = Path(csv_files[0])
    except ImportError:
        raise FileNotFoundError("Place ecommerce_tableau_final.csv beside this notebook or update DATA_PATH.")

df_raw = pd.read_csv(DATA_PATH)
print("Loaded:", DATA_PATH, "| Raw shape:", df_raw.shape)
display(df_raw.head())

## 1. Data quality audit

In [ ]:
audit = pd.DataFrame({
    "dtype": df_raw.dtypes.astype(str),
    "missing": df_raw.isna().sum(),
    "unique": df_raw.nunique(),
    "missing_pct": (df_raw.isna().mean()*100).round(2)
})
display(audit)
print("Exact duplicate rows:", df_raw.duplicated().sum())
print("Date range:", df_raw["OrderDate"].min(), "to", df_raw["OrderDate"].max())

## 2. Cleaning and feature engineering
Exact duplicates are removed. Date attributes are recomputed from `OrderDate`. Monetary outliers are retained rather than capped because extreme values may represent genuine high-value transactions.

In [ ]:
df = df_raw.copy()
df["OrderDate"] = pd.to_datetime(df["OrderDate"], errors="coerce")
df["ShipDate"] = pd.to_datetime(df["ShipDate"], errors="coerce")
df = df.dropna(subset=["OrderDate","CustomerName","City","Sales"]).drop_duplicates().copy()
for c in ["Sales","Quantity","Discount","Profit","Revenue"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")
df = df.dropna(subset=["Sales","Quantity","Discount","Profit"])
df = df[(df.Sales>=0)&(df.Quantity>=0)&(df.Discount.between(0,1))].copy()
df["OrderYear"] = df.OrderDate.dt.year
df["OrderMonth"] = df.OrderDate.dt.month
df["YearMonth"] = df.OrderDate.dt.to_period("M").astype(str)
df["ShippingDays"] = (df.ShipDate-df.OrderDate).dt.days.clip(lower=0)
print("Clean shape:", df.shape)
display(df.describe(include="all").T)

## 3. Exploratory analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14,5))
sns.histplot(df["Sales"], bins=50, ax=axes[0])
axes[0].set_title("Sales distribution")
sns.boxplot(x=df["Profit"], ax=axes[1])
axes[1].set_title("Profit distribution")
plt.tight_layout()
plt.show()

monthly = df.set_index("OrderDate").resample("MS").agg(
    Sales=("Sales","sum"), Profit=("Profit","sum"),
    Orders=("OrderID","nunique"), Quantity=("Quantity","sum")
).asfreq("MS").fillna(0).reset_index().rename(columns={"OrderDate":"MonthStart"})
plt.figure(figsize=(14,5))
plt.plot(monthly["MonthStart"], monthly["Sales"], marker="o")
plt.title("Monthly sales trend"); plt.xlabel("Month"); plt.ylabel("Sales")
plt.grid(alpha=.25); plt.tight_layout(); plt.show()

display(df.groupby("Category", observed=True).agg(
    Sales=("Sales","sum"), Profit=("Profit","sum"), Orders=("OrderID","nunique")
).sort_values("Sales", ascending=False))

## 4. Customer segmentation using RFM
Recency = days since last purchase; Frequency = distinct orders; Monetary = total sales. Features are log-transformed and standardized before K-Means.

In [ ]:
snapshot = df.OrderDate.max() + pd.Timedelta(days=1)
rfm = df.groupby("CustomerName").agg(
    LastPurchaseDate=("OrderDate","max"),
    Frequency=("OrderID","nunique"),
    Monetary=("Sales","sum"),
    TotalProfit=("Profit","sum"),
    TotalQuantity=("Quantity","sum")
).reset_index()
rfm["Recency"] = (snapshot-rfm["LastPurchaseDate"]).dt.days
X = rfm[["Recency","Frequency","Monetary"]].copy()
for c in X: X[c] = np.log1p(X[c].clip(lower=0))
scaler = StandardScaler()
Xs = scaler.fit_transform(X)
scores=[]
best_model=None; best_score=-1
for k in range(2,7):
    km=KMeans(n_clusters=k, random_state=42, n_init=20)
    labels=km.fit_predict(Xs)
    score=silhouette_score(Xs, labels)
    scores.append({"k":k,"silhouette":score})
    if score>best_score: best_model,best_score=km,score
rfm["Cluster"]=best_model.labels_
display(pd.DataFrame(scores))
print("Selected k:", best_model.n_clusters, "Silhouette:", round(best_score,4))

In [ ]:
segment_summary = rfm.groupby("Cluster").agg(
    Customers=("CustomerName","count"), AvgRecency=("Recency","mean"),
    AvgFrequency=("Frequency","mean"), AvgMonetary=("Monetary","mean")
).reset_index()
segment_summary["ValueIndex"] = (
    segment_summary.AvgFrequency.rank(pct=True) +
    segment_summary.AvgMonetary.rank(pct=True) +
    (1-segment_summary.AvgRecency.rank(pct=True))
)
label_map={}
for _, row in segment_summary.iterrows():
    if row.AvgRecency >= segment_summary.AvgRecency.quantile(.75) and row.AvgFrequency <= segment_summary.AvgFrequency.median():
        label="At Risk"
    elif row.ValueIndex >= segment_summary.ValueIndex.quantile(.75):
        label="Champions"
    elif row.AvgFrequency >= segment_summary.AvgFrequency.median():
        label="Loyal Customers"
    else:
        label="Potential / Occasional"
    label_map[int(row.Cluster)] = label
rfm["CustomerSegment"] = rfm.Cluster.map(label_map)
display(segment_summary.assign(CustomerSegment=segment_summary.Cluster.map(label_map)))
sns.scatterplot(data=rfm, x="Recency", y="Monetary", hue="CustomerSegment")
plt.title("Customer segments: Recency vs Monetary"); plt.tight_layout(); plt.show()

## 5. Monthly sales forecasting
A Random Forest regressor uses lagged monthly sales and rolling averages. The final 12 months are held out for testing. This is a small time series (36 monthly observations), so results and forward estimates must be interpreted cautiously.

In [ ]:
series = monthly.set_index("MonthStart")["Sales"].asfreq("MS").fillna(0)
feat = pd.DataFrame({"Date":series.index, "Sales":series.values})
feat["month"]=feat.Date.dt.month; feat["year"]=feat.Date.dt.year; feat["time_index"]=np.arange(len(feat))
for lag in [1,2,3,6,12]: feat[f"lag_{lag}"]=feat.Sales.shift(lag)
feat["rolling_mean_3"]=feat.Sales.shift(1).rolling(3).mean()
feat["rolling_mean_6"]=feat.Sales.shift(1).rolling(6).mean()
feat=feat.dropna().reset_index(drop=True)
features=[c for c in feat if c not in ["Date","Sales"]]
split_date=series.index[-12]
train=feat[feat.Date<split_date]; test=feat[feat.Date>=split_date]
model=RandomForestRegressor(n_estimators=500,max_depth=5,min_samples_leaf=2,random_state=42,n_jobs=-1)
model.fit(train[features],train.Sales)
pred=model.predict(test[features])
metrics={"MAE":mean_absolute_error(test.Sales,pred),
         "RMSE":mean_squared_error(test.Sales,pred)**.5,
         "R2":r2_score(test.Sales,pred)}
display(pd.DataFrame([metrics]))
plt.figure(figsize=(13,5))
plt.plot(test.Date,test.Sales,label="Actual",marker="o")
plt.plot(test.Date,pred,label="Predicted",marker="o")
plt.legend(); plt.title("Holdout: actual vs predicted monthly sales")
plt.xticks(rotation=45); plt.tight_layout(); plt.show()

## 6. Six-month forward estimate

In [ ]:
final_model=RandomForestRegressor(n_estimators=500,max_depth=5,min_samples_leaf=2,random_state=42,n_jobs=-1)
final_model.fit(feat[features],feat.Sales)
history=list(series.values.astype(float)); future=[]
for date in pd.date_range(series.index[-1]+pd.offsets.MonthBegin(1),periods=6,freq="MS"):
    row={"month":date.month,"year":date.year,"time_index":len(history),
         "lag_1":history[-1],"lag_2":history[-2],"lag_3":history[-3],
         "lag_6":history[-6],"lag_12":history[-12],
         "rolling_mean_3":np.mean(history[-3:]),"rolling_mean_6":np.mean(history[-6:])}
    y=max(0,float(final_model.predict(pd.DataFrame([row])[features])[0]))
    future.append({"MonthStart":date,"PredictedSales":y})
    history.append(y)
forecast=pd.DataFrame(future)
display(forecast)
forecast.to_csv(OUTPUT/"data/sales_forecast_6_months.csv",index=False)

## 7. Supervised transaction-level sales estimation
Predict transaction Sales from available order attributes. Profit and Revenue are excluded to prevent target leakage. The split is chronological, not random.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

num_cols=["Quantity","Discount","OrderMonth","OrderYear"]
cat_cols=["ShipMode","Segment","Region","Category","SubCategory","ProductName"]
model_features=num_cols+cat_cols
ordered=df.sort_values("OrderDate").reset_index(drop=True)
cut=int(len(ordered)*.8)
train_orders=ordered.iloc[:cut]; test_orders=ordered.iloc[cut:]
prep=ColumnTransformer([
    ("num",Pipeline([("imputer",SimpleImputer(strategy="median")),("scale",StandardScaler())]),num_cols),
    ("cat",Pipeline([("imputer",SimpleImputer(strategy="most_frequent")),("onehot",OneHotEncoder(handle_unknown="ignore"))]),cat_cols)
])
sales_pipeline=Pipeline([("preprocess",prep),("regressor",RandomForestRegressor(
    n_estimators=350,min_samples_leaf=2,max_features=.8,random_state=42,n_jobs=-1))])
sales_pipeline.fit(train_orders[model_features],train_orders["Sales"])
order_pred=np.maximum(0,sales_pipeline.predict(test_orders[model_features]))
print("Order-level MAE:", mean_absolute_error(test_orders["Sales"],order_pred))
print("Order-level RMSE:", mean_squared_error(test_orders["Sales"],order_pred)**0.5)
print("Order-level R2:", r2_score(test_orders["Sales"],order_pred))


## 7. Export Power BI datasets and models

In [ ]:
df.to_csv(OUTPUT/"data/ecommerce_cleaned.csv",index=False)
rfm[["CustomerName","Recency","Frequency","Monetary","TotalProfit","TotalQuantity","Cluster","CustomerSegment"]].to_csv(OUTPUT/"data/customer_segments.csv",index=False)
monthly.to_csv(OUTPUT/"data/monthly_sales_actuals.csv",index=False)
bt=test[["Date","Sales"]].copy(); bt["PredictedSales"]=pred; bt["YearMonth"]=bt.Date.dt.strftime("%Y-%m")
bt.rename(columns={"Date":"MonthStart"}).to_csv(OUTPUT/"data/sales_forecast_backtest.csv",index=False)
joblib.dump({"model":best_model,"scaler":scaler,"labels_map":label_map},OUTPUT/"models/customer_segmentation.joblib")
joblib.dump({"model":final_model,"feature_columns":features},OUTPUT/"models/monthly_sales_forecast.joblib")
print("Exports written to:", OUTPUT.resolve())

# Export transaction-level predictions and model
test_export = test_orders[["OrderID", "OrderDate", "CustomerName", "Category", "Sales"]].copy()
test_export["PredictedSales"] = order_pred
test_export.to_csv(OUTPUT / "data/order_sales_predictions.csv", index=False)
joblib.dump(sales_pipeline, OUTPUT / "models/order_sales_prediction.joblib")

model_metrics = {
    "monthly_forecast_random_forest": {
        "MAE": float(metrics["MAE"]),
        "RMSE": float(metrics["RMSE"]),
        "R2": float(metrics["R2"])
    },
    "transaction_sales_random_forest": {
        "MAE": float(mean_absolute_error(test_orders["Sales"], order_pred)),
        "RMSE": float(mean_squared_error(test_orders["Sales"], order_pred) ** 0.5),
        "R2": float(r2_score(test_orders["Sales"], order_pred))
    }
}
with open(OUTPUT / "reports/model_metrics.json", "w") as f:
    json.dump(model_metrics, f, indent=2)

print("\nAll outputs saved under:", OUTPUT.resolve())
print("Download the outputs folder from the Colab Files panel, or zip it with the next cell.")


In [ ]:
# Optional: create one ZIP containing all generated datasets, reports, and models
import shutil
zip_path = shutil.make_archive("ecommerce_analytics_outputs", "zip", OUTPUT)
print("Created:", zip_path)

# In Google Colab, uncomment the next two lines to download it automatically
# from google.colab import files
# files.download(zip_path)
